# Getting started — Infant Gut Shotgun-Metagenome Catalog (v1.2.1)
Load the wide table, define the catalog scope on `age_scope`, filter by confidence and route, look at Sandpiper taxonomy columns, join runs, and plot coverage. Files are expected in the same folder as this notebook.

In [1]:
import pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
wide = pd.read_parquet('sample_metadata_wide.parquet')
studies = pd.read_parquet('study_metadata_wide.parquet')
print(wide.shape, studies.shape)
wide.head(3).T.head(30)

(154206, 147) (389, 117)


,0,1,2
sample_key,SAMD00002748,SAMD00002749,SAMD00002750
study_accession,PRJDA72415,PRJDA72415,PRJDA72415
secondary_sample,DRS012463,DRS012478,DRS012449
sample_title,TYMC108-feces_Distant,TYMC116-feces_Acute,TYMC101-feces_Distant
body_site_class,primary,primary,primary
collection_date,NaN,NaN,NaN
is_gold_heldout,False,False,False
probiotic_exposure,NaN,NaN,NaN
preterm_status,NaN,NaN,NaN
gestational_age_weeks,NaN,NaN,NaN


## 1. Scope: age-evidenced infant samples
`age_scope ∈ {infant_evidenced, study_all_infant}` is the age scope; `catalog_scope` additionally requires a gut/unknown body site (`body_site_class ∈ {primary, unknown}`) and is the headline scope. `adult_flagged` means an over-range age (> 1,100 d, includes children).

In [1]:
print(wide.age_scope.value_counts().to_string())
age_infant = wide[wide.age_scope.isin(['infant_evidenced', 'study_all_infant'])]
infant = wide[wide.catalog_scope]
print(len(age_infant), 'age-scope infant samples;', len(infant), 'catalog-scope (age ∧ body site)')
aged = infant[infant.age_at_collection_days.notna() & infant['age_at_collection_days__route'].isin(['R1', 'R2'])]
print(len(aged), 'with per-sample age from archive/table evidence')
fig, ax = plt.subplots(figsize=(6, 3))
aged.age_at_collection_days.clip(upper=1100).plot.hist(bins=60, ax=ax); ax.set_xlabel('age at collection (days)')
fig.tight_layout(); fig.savefig('getting_started_age_hist.png', dpi=100); plt.close(fig)

age_scope
infant_evidenced                 67598
age_unknown_no_study_estimate    49556
age_unknown_mixed_study          14410
adult_flagged                     9559
study_all_infant                  7097
non_infant_role                   5986
74695 age-scope infant samples; 71795 catalog-scope (age ∧ body site)
46861 with per-sample age from archive/table evidence


## 2. A study-level view: which cohorts have delivery mode AND feeding for most samples?

In [1]:
good = studies[(studies.cov_delivery_mode >= 0.8) & (studies.cov_feeding_mode >= 0.8)][['study_accession', 'study_title', 'n_sample_rows', 'n_catalog_scope', 'cov_age_at_collection_days', 'cov_delivery_mode', 'cov_feeding_mode', 'cohort_name']]
good.sort_values('n_catalog_scope', ascending=False).head(15)

,study_accession,study_title,n_sample_rows,n_catalog_scope,cov_age_at_collection_days,cov_delivery_mode,cov_feeding_mode,cohort_name
38,PRJEB32631,Early_life_microbiota_colonisation,1679,1679,0.998213,1.000000,0.807624,Shao 2019 UK
73,PRJEB62690,"Infant stool, 5-HMO supplemented formula vs. control formula vs. breastfed",1110,1110,1.000000,1.000000,1.000000,singleton study PRJEB62690
195,PRJNA1404625,Investigating the role of the infant gut microbiome in rotavirus vaccine efficacy,835,835,1.000000,0.990419,0.928144,singleton study PRJNA1404625
44,PRJEB39610,Longitudinal metagenomic analysis of the gut microbiome in preterm infants diagnosed with necrotising enterocolitis and matched controls,644,644,1.000000,1.000000,0.972050,NaN
339,PRJNA807448,The Antibiotics and Immune Responses Study,459,459,1.000000,1.000000,0.993464,Ryan 2025 Australia
266,PRJNA489090,"Human infant gut metagenome, infant shotgun sequencing",429,429,1.000000,1.000000,0.927739,singleton study PRJNA489090
259,PRJNA473126,Infant Diet and Maternal Gestational Weight Gain Influence Functional Maturation of the Infant Gut Microbiome,402,402,1.000000,1.000000,1.000000,Baumann-Dudenhoeffer 2018 USA
248,PRJNA396794,preterm infant gut metagenomes (NIH Y4 Cohort),305,305,1.000000,0.996721,0.993443,NIH Y4 Cohort
282,PRJNA549787,"South African HIV-exposed, uninfected infant gut microbiomes",165,165,1.000000,0.993939,1.000000,D'Souza 2020 South Africa
224,PRJNA294605,Metagenomes from 11 human infant fecal samples hospitalized in the same intensive care unit,158,158,0.892405,0.892405,0.892405,NaN


## 3. Build an analysis slice and export it
Example: term, vaginally born, exclusively breastfed infants sampled before 6 months, with confidence ≥ 0.7 on every field used.

In [1]:
sl = infant[(infant.preterm_status == 'term') & (infant.delivery_mode == 'vaginal') & (infant.feeding_mode == 'exclusive_breast') & (infant.age_at_collection_days < 183)]
for f in ['preterm_status', 'delivery_mode', 'feeding_mode', 'age_at_collection_days']:
    sl = sl[sl[f + '__confidence'] >= 0.7]
print(len(sl), 'samples in', sl.study_accession.nunique(), 'studies')
sl[['sample_key', 'study_accession', 'run_accessions', 'age_at_collection_days', 'delivery_mode', 'feeding_mode']].to_csv('my_slice.csv', index=False)

1181 samples in 16 studies


## 4. Sandpiper taxonomy (SingleM community profiles, GTDB R232)
`sp_profiled` marks samples with a profile; `sp_ra_*` are fractions of prokaryotic coverage (≈ cell proportions). Genus indicators include GTDB alphabetic-suffix genera (e.g. Enterococcus_B = E. faecium). Exclude `sp_low_depth` rows for composition statistics.

In [1]:
prof = infant[infant.sp_profiled & ~infant.sp_low_depth.fillna(False).astype(bool)]
print(len(prof), 'profiled catalog-scope samples with adequate depth in', prof.study_accession.nunique(), 'studies')
print('median Bifidobacterium RA:', round(prof.sp_ra_g_Bifidobacterium.median(), 3))
fig, ax = plt.subplots(figsize=(6, 3))
prof.sp_ra_g_Bifidobacterium.plot.hist(bins=40, ax=ax); ax.set_xlabel('Bifidobacterium relative abundance (fraction of prokaryotic coverage)')
fig.tight_layout(); fig.savefig('getting_started_bifido_hist.png', dpi=100); plt.close(fig)
sl_prof = sl[sl.sp_profiled]
sl_prof[['sample_key', 'sp_ra_g_Bifidobacterium', 'sp_ra_enterobacterales_core', 'sp_shannon_genus', 'sandpiper_url']].head()

41290 profiled catalog-scope samples with adequate depth in 181 studies
median Bifidobacterium RA: 0.093


,sample_key,sp_ra_g_Bifidobacterium,sp_ra_enterobacterales_core,sp_shannon_genus,sandpiper_url
18396,SAMEA115454978,0.931955,0.011937,0.365845,https://sandpiper.qut.edu.au/run/ERR12814313
18397,SAMEA115454979,0.955924,0.000000,0.240661,https://sandpiper.qut.edu.au/run/ERR12812857
18408,SAMEA115454991,0.943853,0.000000,0.267136,https://sandpiper.qut.edu.au/run/ERR12814168
18412,SAMEA115454995,0.040936,0.000000,3.729486,https://sandpiper.qut.edu.au/run/ERR12812864
18417,SAMEA115455000,0.746375,0.001359,0.950627,https://sandpiper.qut.edu.au/run/ERR12814353


## 5. Trace a value back to its evidence

In [1]:
det = pd.read_parquet('sample_determinations.parquet')
k = sl.sample_key.iloc[0]
det[det.sample_key == k][['field_name', 'value_normalized', 'confidence', 'route', 'evidence_source', 'evidence_locator', 'evidence_quote']]

,field_name,value_normalized,confidence,route,evidence_source,evidence_locator,evidence_quote
61731,age_at_collection_days,8,0.85,R2,paper.supp.table,PMC11183301/mmc2.xlsx/Tab1!timepoint_months:57,timepoint_months=0.25
61732,antibiotic_exposure,no,0.85,R2,paper.supp.table,PMC12222458/41467_2025_61154_MOESM8_ESM.xlsx/metadata_infants!AB_BB:112,AB_BB=no
61733,birth_weight_grams,3150,0.85,R2,paper.supp.table,PMC11183301/mmc2.xlsx/Tab1!birth_weight (Kg):57,birth_weight (Kg)=3.15
61734,country,ES,0.90,R1,biosample_attr,country,Spain
61735,delivery_mode,vaginal,0.85,R2,paper.supp.table,PMC11183301/mmc2.xlsx/Tab1!delivery:57,delivery=vaginal
61736,feeding_mode,exclusive_breast,0.85,R2,paper.supp.table,PMC11183301/mmc2.xlsx/Tab1!feeding_cat:57,feeding_cat=Breastfeeding
61737,gestational_age_weeks,39.0,0.85,R2,paper.supp.table,PMC11183301/mmc2.xlsx/Tab1!gestational_age:57,gestational_age=39.0
61738,maternal_antibiotics,no,0.85,R2,paper.supp.table,PMC11183301/mmc2.xlsx/Tab1!intrapartum_ATB:57,intrapartum_ATB=0.0
61739,multiple_birth,singleton,0.85,R2,paper.supp.table,PMC11183301/mmc2.xlsx/Tab1!Twin:57,Twin=0.0
61740,preterm_status,term,0.80,R3,paper.fulltext.methods,PMC12222458/methods,"66 of healthy, full-term mother-infant pairs"


## 6. Coverage per field (age-scope and catalog-scope denominators)

In [1]:
cov = pd.read_csv('field_coverage_summary.csv', index_col=0)
fig, ax = plt.subplots(figsize=(6, 4))
cov[['coverage_age_scope_infant', 'coverage_catalog_scope']].sort_values('coverage_catalog_scope').plot.barh(ax=ax); ax.set_xlabel('fraction of samples with a value')
fig.tight_layout(); fig.savefig('getting_started_coverage.png', dpi=100); plt.close(fig)
cov[['coverage_age_scope_infant', 'coverage_catalog_scope']].round(3)

,coverage_age_scope_infant,coverage_catalog_scope
field,,
probiotic_exposure,0.037,0.037
preterm_status,0.434,0.451
gestational_age_weeks,0.165,0.171
delivery_mode,0.314,0.321
feeding_mode,0.158,0.158
antibiotic_exposure,0.174,0.175
age_at_collection_days,0.718,0.733
birth_weight_grams,0.140,0.145
country,0.960,0.958


## 7. Get the runs for download (e.g. `fasterq-dump` / ENA FTP)
Join on `biosample_accession` for BioSample units and on `run_accession` for run units (`sample_unit = 'run'`).

In [1]:
runs = pd.read_parquet('runs.parquet')
bs_units = sl[sl.sample_unit == 'biosample'].biosample_accession
run_units = sl[sl.sample_unit == 'run'].run_accession
my_runs = runs[runs.sample_accession.isin(bs_units) | runs.run_accession.isin(run_units)]
print(len(my_runs), 'runs for', len(sl), 'samples')
my_runs[['run_accession', 'sample_accession', 'study_accession', 'library_layout', 'instrument_model', 'read_count']].head()

1181 runs for 1181 samples


,run_accession,sample_accession,study_accession,library_layout,instrument_model,read_count
9587,ERR12812864,SAMEA115454995,PRJEB74322,PAIRED,Illumina HiSeq 2500,18122268
9595,ERR12813208,SAMEA115455060,PRJEB74322,PAIRED,Illumina HiSeq 2500,32203140
9596,ERR12813219,SAMEA115455027,PRJEB74322,PAIRED,Illumina HiSeq 2500,23039418
9597,ERR12814168,SAMEA115454991,PRJEB74322,PAIRED,Illumina HiSeq 2500,16857338
9598,ERR12814185,SAMEA115455030,PRJEB74322,PAIRED,Illumina HiSeq 2500,23809548
